# Comparing LSTM and GRU for Sequence Forecasting

This notebook uses a small synthetic product-price sequence to demonstrate:

- Sliding-window preparation for recurrent models
- Training a compact LSTM regressor
- Checking a next-step prediction
- Comparing the same task with a GRU
- Reusing the sequence-preparation function for another time series

The data are illustrative rather than real marketplace data.

## 1. Prepare a time-series dataset

In [ ]:

import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(26)

# Illustrative daily prices for one fictional product.
daily_prices = np.array(
    [745, 760, 752, 775, 790, 784, 805, 820, 812, 835, 848, 842],
    dtype=np.float32
)

lookback = 4

def sliding_windows(series, length):
    inputs, targets = [], []
    for position in range(len(series) - length):
        inputs.append(series[position:position + length])
        targets.append(series[position + length])
    return np.asarray(inputs, dtype=np.float32), np.asarray(targets, dtype=np.float32)

X, y = sliding_windows(daily_prices, lookback)
X = X[..., np.newaxis]

print("Number of samples:", X.shape[0])
print("Model input shape:", X.shape)
print("Target shape:", y.shape)


## 2. Train an LSTM regression model

In [ ]:

lstm_net = keras.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.LSTM(24),
    layers.Dense(12, activation="tanh"),
    layers.Dense(1)
], name="price_lstm")

lstm_net.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.002),
    loss="mse"
)

lstm_net.fit(
    X,
    y,
    epochs=18,
    batch_size=2,
    verbose=0
)

print("LSTM training complete.")


## 3. Inspect the final LSTM prediction

In [ ]:

recent_window = X[-1:]
expected_value = y[-1]

lstm_output = float(lstm_net.predict(recent_window, verbose=0).squeeze())

print(f"LSTM estimate: {lstm_output:.2f}")
print(f"Reference value: {expected_value:.2f}")
print(f"Absolute difference: {abs(lstm_output - expected_value):.2f}")


## 4. Train an equivalent GRU model

In [ ]:

gru_net = keras.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.GRU(24),
    layers.Dense(12, activation="tanh"),
    layers.Dense(1)
], name="price_gru")

gru_net.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.002),
    loss="mse"
)

start = time.perf_counter()
gru_net.fit(
    X,
    y,
    epochs=18,
    batch_size=2,
    verbose=0
)
elapsed = time.perf_counter() - start

gru_output = float(gru_net.predict(recent_window, verbose=0).squeeze())

print(f"GRU estimate: {gru_output:.2f}")
print(f"Reference value: {expected_value:.2f}")
print(f"GRU training time: {elapsed:.4f} seconds")


## 5. Reuse the sliding-window helper

The same preparation method can be used for other sequential data, such as weekly music-play counts.

In [ ]:

weekly_plays = [120, 135, 128, 150, 162, 158, 175, 184, 179]
play_inputs, play_targets = sliding_windows(weekly_plays, length=3)

print("Play-count inputs:")
print(play_inputs)
print("Play-count targets:")
print(play_targets)


### Main observations

- A recurrent model receives a sequence shaped as **samples × time steps × features**.
- LSTM and GRU layers both process ordered data while maintaining internal state.
- GRUs use a simpler gating design than LSTMs.
- Training time depends on hardware, sequence length, model size, and framework overhead.
- A tiny synthetic dataset is useful for demonstrating mechanics, but it is not enough to establish real forecasting accuracy.